# Conformal calibration on the sinusoidal dataset

This notebook loads the checked-in conditional normalizing-flow predictor, calibrates an $L_2$ latent region from the calibration split, and evaluates it on a separate held-out split. It visualizes conditional prediction contours, target-space volume, coverage as a function of the covariate, and nominal-versus-empirical coverage.

The dataset API calls the held-out split `test`; below it is named `validation_data` because it is used only for post-calibration validation.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import torch

repo_root = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "pyproject.toml").is_file()
    ),
    None,
)
if repo_root is None:
    raise RuntimeError("Could not locate the repository root.")

src_path = str(repo_root / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from configs.calibrators import NormCalibratorConfig
from configs.conformal import ConformalPredictorConfig
from configs.datasets.synthetic import SinusoidalTransportDatasetConfig
from conformal import ConformalPredictor
from data.datasets.synthetic import SinusoidalTransportDataset
from data.loaders import make_xy_dataloader
from predictors.transport import NormalizingFlowPredictor

plt.rcParams.update({
    "figure.dpi": 130,
    "savefig.dpi": 250,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

In [ ]:
data_seed = 23  # Different from the seed used by the checked-in checkpoint.
plot_seed = 7
torch.manual_seed(plot_seed)
np.random.seed(plot_seed)

device = "cpu"
dtype = "float32"

# The predictor is loaded, so the train split is generated only for a complete
# dataset object and is not used below.
n_train = 1_000
n_calibration = 2_048
n_validation = 4_096
calibration_batch_size = 256
validation_batch_size = 512

coverage_mass = 0.90
coverage_grid = np.array([0.50, 0.70, 0.80, 0.90, 0.95])
selected_x_values = np.array([-0.75, 0.0, 0.75])
contour_x_half_width = 0.12
number_of_x_bins = 10

number_of_boundary_points = 512
boundary_batch_size = 256
volume_mc_samples = 2_048
volume_batch_size = 1_024

checkpoint_path = repo_root / "weights" / "normalizing_flow_predictor.pt"
print("checkpoint:", checkpoint_path)

## Fresh sinusoidal calibration and validation data

Calibration and evaluation use newly generated observations. The calibration loader is deliberately batched to exercise `ConformalPredictor.fit(dataloader)`.

In [ ]:
dataset_config = SinusoidalTransportDatasetConfig(
    n_train=n_train,
    n_calibration=n_calibration,
    n_test=n_validation,
    x_dim=1,
    y_dim=2,
    seed=data_seed,
    device=device,
    dtype=dtype,
)
dataset = SinusoidalTransportDataset(dataset_config)
splits = dataset.get_splits()

calibration_data = splits.calibration
validation_data = splits.test

calibration_loader = make_xy_dataloader(
    calibration_data,
    batch_size=calibration_batch_size,
    shuffle=False,
)
validation_loader = make_xy_dataloader(
    validation_data,
    batch_size=validation_batch_size,
    shuffle=False,
)

print("calibration x/y:", calibration_data.x.shape, calibration_data.y.shape)
print("validation x/y:", validation_data.x.shape, validation_data.y.shape)
print("x support:", (dataset_config.x_low, dataset_config.x_high))

## Load and calibrate the predictor

The norm calibrator pulls each calibration batch back through the learned transport and selects the exact split-conformal order statistic of the latent Euclidean norms.

In [ ]:
if not checkpoint_path.is_file():
    raise FileNotFoundError(
        f"Missing predictor checkpoint: {checkpoint_path}"
    )

predictor = NormalizingFlowPredictor.load(
    str(checkpoint_path),
    map_location=device,
)
predictor.eval()

if predictor.x_dim != dataset.x_dim or predictor.y_dim != dataset.y_dim:
    raise ValueError("The checkpoint dimensions do not match the dataset.")

print("loaded predictor:", type(predictor).__name__)
print("predictor config:", predictor.config.model_dump())

In [ ]:
conformal_config = ConformalPredictorConfig(
    coverage_mass=coverage_mass,
    calibrator=NormCalibratorConfig(p=2.0),
    volume_mc_samples=volume_mc_samples,
    volume_batch_size=volume_batch_size,
    volume_seed=plot_seed,
)
conformal_predictor = ConformalPredictor(
    predictor=predictor,
    config=conformal_config,
)
conformal_predictor.fit(calibration_loader)

calibrated_threshold = float(
    conformal_predictor.threshold.detach().cpu()
)
print(f"requested coverage mass: {coverage_mass:.3f}")
print(f"calibrated latent radius: {calibrated_threshold:.4f}")

## Held-out coverage

Membership is evaluated in batches on observations that were not used for calibration. Besides the overall marginal coverage, we compute coverage across covariate bins and repeat calibration at several nominal masses.

In [ ]:
def membership_mask(model, dataloader):
    masks = []
    for x_batch, y_batch in dataloader:
        masks.append(model.contains(x=x_batch, y=y_batch).detach().cpu())
    return torch.cat(masks, dim=0)


validation_inside = membership_mask(conformal_predictor, validation_loader)
validation_x = validation_data.x.detach().cpu()
validation_y = validation_data.y.detach().cpu()
empirical_coverage = float(validation_inside.to(torch.float32).mean())

x_bin_edges = np.linspace(
    dataset_config.x_low,
    dataset_config.x_high,
    number_of_x_bins + 1,
)
x_bin_centers = 0.5 * (x_bin_edges[:-1] + x_bin_edges[1:])
x_bin_index = np.digitize(
    validation_x[:, 0].numpy(),
    x_bin_edges[1:-1],
)
coverage_by_x = np.full(number_of_x_bins, np.nan)
coverage_standard_error = np.full(number_of_x_bins, np.nan)
validation_count_by_x = np.zeros(number_of_x_bins, dtype=int)

inside_numpy = validation_inside.numpy()
for bin_index in range(number_of_x_bins):
    in_bin = x_bin_index == bin_index
    count = int(in_bin.sum())
    validation_count_by_x[bin_index] = count
    if count > 0:
        bin_coverage = float(inside_numpy[in_bin].mean())
        coverage_by_x[bin_index] = bin_coverage
        coverage_standard_error[bin_index] = np.sqrt(
            bin_coverage * (1.0 - bin_coverage) / count
        )

coverage_curve = []
for nominal_mass in coverage_grid:
    if np.isclose(nominal_mass, coverage_mass):
        candidate = conformal_predictor
        candidate_inside = validation_inside
    else:
        candidate = ConformalPredictor(
            predictor=predictor,
            config=ConformalPredictorConfig(
                coverage_mass=float(nominal_mass),
                calibrator=NormCalibratorConfig(p=2.0),
            ),
        ).fit(calibration_loader)
        candidate_inside = membership_mask(candidate, validation_loader)

    coverage_curve.append({
        "nominal": float(nominal_mass),
        "empirical": float(candidate_inside.to(torch.float32).mean()),
        "threshold": float(candidate.threshold.detach().cpu()),
    })

print(f"held-out empirical coverage: {empirical_coverage:.4f}")
print("\nNominal  Empirical  Latent radius")
for result in coverage_curve:
    print(
        f"{result['nominal']:7.2f}  "
        f"{result['empirical']:9.4f}  "
        f"{result['threshold']:13.4f}"
    )

## Calibrated contours and target-space volume

The calibrated threshold is a latent radius. Its circle is pushed through the learned conditional transport at selected covariate values. Volume is estimated from the learned forward Jacobian over the same latent ball.

In [ ]:
def close_contour(points):
    points = np.asarray(points)
    if np.allclose(points[0], points[-1]):
        return points
    return np.vstack([points, points[0]])


def repeated_x(x_value, batch_size):
    return torch.full(
        (batch_size, dataset.x_dim),
        float(x_value),
        device=predictor.device,
        dtype=predictor.dtype,
    )


def learned_contour(x_value, latent_boundary):
    mapped = []
    for start in range(0, latent_boundary.shape[0], boundary_batch_size):
        u_batch = latent_boundary[start:start + boundary_batch_size]
        x_batch = repeated_x(x_value, u_batch.shape[0])
        with torch.no_grad():
            mapped.append(
                conformal_predictor.pushforward(x=x_batch, u=u_batch)
                .detach()
                .cpu()
            )
    return close_contour(torch.cat(mapped, dim=0).numpy())


def oracle_contour(x_value, latent_boundary):
    mapped = []
    for start in range(0, latent_boundary.shape[0], boundary_batch_size):
        u_batch = latent_boundary[start:start + boundary_batch_size]
        x_batch = repeated_x(x_value, u_batch.shape[0])
        mapped.append(
            dataset.push_u_given_x(u=u_batch, x=x_batch).detach().cpu()
        )
    return close_contour(torch.cat(mapped, dim=0).numpy())


theta = torch.linspace(
    0.0,
    2.0 * torch.pi,
    number_of_boundary_points + 1,
    device=predictor.device,
    dtype=predictor.dtype,
)
latent_boundary = calibrated_threshold * torch.stack(
    [torch.cos(theta), torch.sin(theta)],
    dim=-1,
)

learned_contours = {
    float(x_value): learned_contour(x_value, latent_boundary)
    for x_value in selected_x_values
}
oracle_contours = {
    float(x_value): oracle_contour(x_value, latent_boundary)
    for x_value in selected_x_values
}

volume_x = torch.linspace(
    dataset_config.x_low,
    dataset_config.x_high,
    41,
    device=predictor.device,
    dtype=predictor.dtype,
).unsqueeze(-1)
estimated_volume = conformal_predictor.volume(
    volume_x,
    number_of_samples=volume_mc_samples,
    batch_size=volume_batch_size,
    seed=plot_seed,
).detach().cpu().numpy()

selected_x_tensor = torch.as_tensor(
    selected_x_values,
    device=predictor.device,
    dtype=predictor.dtype,
).unsqueeze(-1)
selected_volume = conformal_predictor.volume(
    selected_x_tensor,
    number_of_samples=volume_mc_samples,
    batch_size=volume_batch_size,
    seed=plot_seed,
).detach().cpu().numpy()

# The true sinusoidal map has determinant equal to its vertical scale.
volume_x_numpy = volume_x[:, 0].detach().cpu().numpy()
x_midpoint = 0.5 * (dataset_config.x_low + dataset_config.x_high)
oracle_vertical_scale = dataset_config.vertical_scale * np.exp(
    dataset_config.vertical_scale_x_scale * (volume_x_numpy - x_midpoint)
)
oracle_volume = (
    np.pi * calibrated_threshold**2 * oracle_vertical_scale
)

In [ ]:
validation_x_numpy = validation_x[:, 0].numpy()
validation_y_numpy = validation_y.numpy()
rng = np.random.default_rng(plot_seed)

fig, axes = plt.subplots(
    1,
    len(selected_x_values),
    figsize=(13.2, 4.1),
    constrained_layout=True,
)

for panel_index, (axis, x_value, region_volume) in enumerate(
    zip(axes, selected_x_values, selected_volume)
):
    learned = learned_contours[float(x_value)]
    oracle = oracle_contours[float(x_value)]
    near_x = np.abs(validation_x_numpy - x_value) <= contour_x_half_width
    nearby_indices = np.flatnonzero(near_x)
    if nearby_indices.size > 300:
        nearby_indices = np.sort(
            rng.choice(nearby_indices, size=300, replace=False)
        )

    nearby_inside = inside_numpy[nearby_indices]
    nearby_y = validation_y_numpy[nearby_indices]
    local_coverage = float(inside_numpy[near_x].mean())

    axis.fill(
        learned[:, 0],
        learned[:, 1],
        color="royalblue",
        alpha=0.10,
    )
    axis.plot(
        learned[:, 0],
        learned[:, 1],
        color="royalblue",
        linewidth=2.2,
        label="learned calibrated contour" if panel_index == 0 else None,
    )
    axis.plot(
        oracle[:, 0],
        oracle[:, 1],
        color="darkorange",
        linestyle="--",
        linewidth=1.8,
        label="oracle contour" if panel_index == 0 else None,
    )
    axis.scatter(
        nearby_y[nearby_inside, 0],
        nearby_y[nearby_inside, 1],
        s=11,
        color="forestgreen",
        alpha=0.45,
        linewidths=0,
        label="held-out: inside" if panel_index == 0 else None,
    )
    axis.scatter(
        nearby_y[~nearby_inside, 0],
        nearby_y[~nearby_inside, 1],
        s=18,
        color="firebrick",
        alpha=0.70,
        marker="x",
        linewidths=0.8,
        label="held-out: outside" if panel_index == 0 else None,
    )

    contour_points = np.vstack([learned, oracle])
    lower = contour_points.min(axis=0)
    upper = contour_points.max(axis=0)
    span = np.maximum(upper - lower, 1e-6)
    axis.set_xlim(lower[0] - 0.12 * span[0], upper[0] + 0.12 * span[0])
    axis.set_ylim(lower[1] - 0.12 * span[1], upper[1] + 0.12 * span[1])
    axis.set_aspect("equal", adjustable="box")
    axis.set_xlabel(r"$y_1$")
    axis.set_ylabel(r"$y_2$")
    axis.set_title(
        rf"$x={x_value:.2f}$" + "\n"
        f"coverage={local_coverage:.3f}, volume={region_volume:.2f}"
    )
    axis.grid(alpha=0.15)

axes[0].legend(frameon=False, fontsize=8, loc="upper left")
fig.suptitle(
    "Calibrated conditional contours "
    f"(target mass={coverage_mass:.2f}, latent radius={calibrated_threshold:.3f})"
)
plt.show()

In [ ]:
nominal_coverage = np.array([result["nominal"] for result in coverage_curve])
observed_coverage = np.array([result["empirical"] for result in coverage_curve])

fig, axes = plt.subplots(1, 3, figsize=(14.2, 4.0), constrained_layout=True)

axes[0].plot(
    volume_x_numpy,
    estimated_volume,
    color="royalblue",
    linewidth=2.2,
    label="learned region",
)
axes[0].plot(
    volume_x_numpy,
    oracle_volume,
    color="darkorange",
    linestyle="--",
    linewidth=1.8,
    label="oracle map of same ball",
)
axes[0].set(
    xlabel=r"$x$",
    ylabel="target-space volume",
    title="Calibrated region volume",
)
axes[0].legend(frameon=False, fontsize=8)
axes[0].grid(alpha=0.20)

valid_bins = np.isfinite(coverage_by_x)
axes[1].errorbar(
    x_bin_centers[valid_bins],
    coverage_by_x[valid_bins],
    yerr=1.96 * coverage_standard_error[valid_bins],
    color="forestgreen",
    marker="o",
    linewidth=1.7,
    capsize=3,
    label="held-out estimate (95% normal CI)",
)
axes[1].axhline(
    coverage_mass,
    color="black",
    linestyle="--",
    linewidth=1.2,
    label="requested marginal coverage",
)
coverage_floor = max(
    0.0,
    min(float(np.nanmin(coverage_by_x)), coverage_mass) - 0.10,
)
axes[1].set(
    xlabel=r"$x$ bin center",
    ylabel="empirical coverage",
    ylim=(coverage_floor, 1.01),
    title=(
        "Coverage across covariates\n"
        f"overall={empirical_coverage:.3f}, target={coverage_mass:.3f}"
    ),
)
axes[1].legend(frameon=False, fontsize=8)
axes[1].grid(alpha=0.20)

diagonal_limits = (0.45, 1.0)
axes[2].plot(
    diagonal_limits,
    diagonal_limits,
    color="0.35",
    linestyle="--",
    linewidth=1.3,
    label="ideal calibration",
)
axes[2].plot(
    nominal_coverage,
    observed_coverage,
    color="purple",
    marker="o",
    linewidth=2.0,
    label="held-out coverage",
)
axes[2].set(
    xlim=diagonal_limits,
    ylim=diagonal_limits,
    xlabel="nominal coverage mass",
    ylabel="empirical coverage",
    title="Marginal calibration curve",
)
axes[2].legend(frameon=False, fontsize=8)
axes[2].grid(alpha=0.20)

plt.show()

## Interpretation

- The contour boundary is the learned pushforward of the empirically calibrated latent radius, not an analytic Gaussian quantile.
- Split conformal calibration targets marginal held-out coverage. The binned plot is a conditional diagnostic and is not expected to equal the target in every bin.
- The volume curve integrates the learned forward Jacobian over the calibrated latent ball. The orange reference uses the known Jacobian of the synthetic sinusoidal transport.
- Change `coverage_mass` to visualize another primary region; the calibration-curve cell already evaluates several nominal levels.